# Keras

In [ ]:
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense


In [ ]:
# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
 X, y = list(), list()
 for i in range(len(sequence)):
  n=i+n_steps # find the end of pattern
  if(n>len(sequence)-1):
    break
  seq_x,seq_y=sequence[i:n], sequence[n]
  X.append(seq_x)
  y.append(seq_y)
 return array(X), array(y)

In [ ]:



# define input sequence
raw_seq = [65,66,64,60,50,55,52,54,60,67,65,64]
# choose a number of time steps
n_steps = 3
# split into samples
X, y = split_sequence(raw_seq, n_steps)
# reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))
print(X)
print(y)


[[[65]
  [66]
  [64]]

 [[66]
  [64]
  [60]]

 [[64]
  [60]
  [50]]

 [[60]
  [50]
  [55]]

 [[50]
  [55]
  [52]]

 [[55]
  [52]
  [54]]

 [[52]
  [54]
  [60]]

 [[54]
  [60]
  [67]]

 [[60]
  [67]
  [65]]]
[60 50 55 52 54 60 67 65 64]


In [ ]:
# define model
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(n_steps, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
# fit model
model.fit(X, y, epochs=200, verbose=0)
# demonstrate prediction
x_input = array([55,58,59])
x_input = x_input.reshape((1, n_steps, n_features))
yhat = model.predict(x_input, verbose=0)
print(yhat)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[[61.74465]]


# PyTorch LSTM

In [ ]:
import torch

# Convert X to a PyTorch tensor
X_tensor = torch.from_numpy(X).float()

# Convert y to a PyTorch tensor
y_tensor = torch.from_numpy(y).float()

# Convert x_input to a PyTorch tensor
x_input_tensor = torch.from_numpy(x_input).float()

# Print shapes and data types to verify
print(f"X_tensor shape: {X_tensor.shape}, dtype: {X_tensor.dtype}")
print(f"y_tensor shape: {y_tensor.shape}, dtype: {y_tensor.dtype}")
print(f"x_input_tensor shape: {x_input_tensor.shape}, dtype: {x_input_tensor.dtype}")

X_tensor shape: torch.Size([9, 3, 1]), dtype: torch.float32
y_tensor shape: torch.Size([9]), dtype: torch.float32
x_input_tensor shape: torch.Size([1, 3, 1]), dtype: torch.float32


In [ ]:
import torch.nn as nn

# Define PyTorch LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # h_n shape: (num_layers * num_directions, batch_size, hidden_size)
        # We need the hidden state from the last layer (assuming 1 layer LSTM) and reshape it
        # For batch_first=True, h_n[-1] gives the hidden state of the last layer for all batches
        out = self.linear(h_n[-1])
        return out

# Instantiate the model with appropriate parameters
# n_features is 1 for univariate time series
# The hidden_size (LSTM units) was 50 in Keras model
# The output_size is 1 for single step prediction
model_pt = LSTMModel(input_size=n_features, hidden_size=50, output_size=1)
print(model_pt)

LSTMModel(
  (lstm): LSTM(1, 50, batch_first=True)
  (linear): Linear(in_features=50, out_features=1, bias=True)
)


In [ ]:
import torch.optim as optim

# Define Loss function and Optimizer
criterion = nn.MSELoss() # Keras used 'mse', so MSELoss is appropriate
optimizer = optim.Adam(model_pt.parameters(), lr=0.001) # Learning rate can be adjusted

# Set model to training mode
model_pt.train()

# Training loop
epochs = 200 # Similar to the Keras model
for epoch in range(epochs):
    # Forward pass
    outputs = model_pt(X_tensor)

    # Calculate loss. Reshape y_tensor to match outputs shape (batch_size, 1)
    loss = criterion(outputs, y_tensor.view(-1, 1))

    # Zero out the gradients
    optimizer.zero_grad()

    # Backward pass
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print loss every 20 epochs to monitor progress
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Training complete.")

Epoch [20/200], Loss: 3402.2661
Epoch [40/200], Loss: 3274.3027
Epoch [60/200], Loss: 3079.1340
Epoch [80/200], Loss: 2908.7556
Epoch [100/200], Loss: 2771.2588
Epoch [120/200], Loss: 2617.1055
Epoch [140/200], Loss: 2467.2725
Epoch [160/200], Loss: 2313.3977
Epoch [180/200], Loss: 2154.2576
Epoch [200/200], Loss: 2020.4227
Training complete.


In [ ]:
import numpy as np

# 1. Set the PyTorch model to evaluation mode
model_pt.eval()

# 2. Disable gradient calculations for prediction
with torch.no_grad():
    # 3. Make a prediction using the trained model_pt on the x_input_tensor
    prediction_pt = model_pt(x_input_tensor)

# 4. Convert the prediction output from a PyTorch tensor to a NumPy array
#    The output is a tensor like [[value]], so we take the first element and convert to scalar.
prediction_pt_np = prediction_pt.cpu().numpy()

# 5. Print the PyTorch model's prediction
print(f"PyTorch Model Prediction: {prediction_pt_np[0][0]:.4f}")

# 6. (Optional) Compare the PyTorch prediction with the yhat obtained from the Keras model.
#    yhat is already a numpy array from the Keras prediction.
print(f"Keras Model Prediction: {yhat[0][0]:.4f}")

# You can also calculate the absolute difference to see how close they are
print(f"Absolute Difference: {np.abs(prediction_pt_np[0][0] - yhat[0][0]):.4f}")

PyTorch Model Prediction: 14.0460
Keras Model Prediction: 61.7447
Absolute Difference: 47.6986
